In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import json
from req_functions_from_ApproxSP import *
from tqdm import tqdm
from generate_profiles import *
from euclidean import *
from mallows import *
from AccompliceManipulation import *
from impartial import *
import pandas as pd

**Mallows MA**

In [3]:

def generate_MA(num_agents):
    votes_left = []
    votes_right = []

    for i in range(num_agents):
        row_left = [(i + j) % num_agents for j in range(num_agents)]
        votes_left.append(row_left)

        row_right = [(i - j) % num_agents for j in range(num_agents)]
        votes_right.append(row_right)

    return votes_left, votes_right

def generate_mallows_MA_votes(num_agents, phi_m, phi_w, **kwargs):
    """ Mallows on top of Asymmetric instance """
    
    votes_left_array, votes_right_array = generate_MA(num_agents)
    
    # Apply Mallows
    votes_left_array = mallows_votes(votes_left_array, phi_m)
    votes_right_array = mallows_votes(votes_right_array, phi_w)
    
    # No need for .tolist()
    votes_left = votes_left_array
    votes_right = votes_right_array

 
    j = 0
    for row in votes_left:
        for i in range(len(row)):
            votes_left[j][i] += 1
        j+=1
    j = 0
    for row in votes_right:
        for i in range(len(row)):
            votes_right[j][i] += 1
        j += 1
 
 
    return [votes_left, votes_right]
def mallows_vote(vote, phi):
    num_candidates = len(vote)
    raw_vote = generate_mallows_votes(1, num_candidates, phi=phi, weight=0)[0]
    new_vote = [0] * len(vote)
    for i in range(num_candidates):
        new_vote[raw_vote[i]] = vote[i]
    return new_vote
 
 
def mallows_votes(votes, phi):
    for i in range(len(votes)):
        votes[i] = mallows_vote(votes[i], phi)
    return votes
 
def generate_mallows_votes(num_voters, num_candidates, phi=0.5, weight=0, **kwargs):
    if phi is None:
        logging.warning('phi is not defined')
    insertion_probabilites_list = []
    for i in range(1, num_candidates):
        insertion_probabilites_list.append(computeInsertionProbas(i, phi))
    V = []
    for i in range(num_voters):
        vote = mallowsVote(num_candidates, insertion_probabilites_list)
        if weight > 0:
            probability = np.random.random()
            if probability <= weight:
                vote.reverse()
        V += [vote]
    return V
 

def computeInsertionProbas(i, phi):
    probas = (i + 1) * [0]
    for j in range(i + 1):
        probas[j] = pow(phi, (i + 1) - (j + 1))
    return probas


def mallowsVote(m, insertion_probabilites_list):
    vote = [0]
    for i in range(1, m):
        index = weighted_choice(insertion_probabilites_list[i - 1])
        vote.insert(index, i)
    return vote


def weighted_choice(choices):
    total = 0
    for w in choices:
        total = total + w
    r = np.random.uniform(0, total)
    upto = 0.0
    for i, w in enumerate(choices):
        if upto + w >= r:
            return i
        upto = upto + w
    assert False, "Shouldn'process_id get here"

In [4]:
##Mallows MA

data_MA = []
numberMA = []
numberMA_pairs = []
n=4
phi_list=[0, 0.2, 0.4, 0.6, 0.8, 1]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            pref=generate_mallows_MA_votes(n, phi1, phi2)
            mpref=pref[0]
            wpref=pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range(1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count_pairs.append(cnt_pairs)
            count.append(count_woman)
            data_MA.append([phi1, phi2, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count) > 0: 
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberMA.append(number_phi)
    numberMA_pairs.append(number_phi_pairs) 
print(numberMA)
print(numberMA_pairs) 

100%|██████████| 6/6 [00:00<00:00, 24.17it/s]

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.04, 0.0, 0.04, 0.0, 0.2, 0.26], [0.14, 0.08, 0.08, 0.12, 0.24, 0.14], [0.2, 0.14, 0.16, 0.12, 0.08, 0.32], [0.16, 0.16, 0.04, 0.16, 0.12, 0.3], [0.2, 0.3, 0.2, 0.12, 0.04, 0.22]]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.08, 0.0, 0.08, 0.0, 0.2, 0.3], [0.18, 0.08, 0.08, 0.12, 0.28, 0.14], [0.28, 0.18, 0.2, 0.12, 0.08, 0.36], [0.24, 0.2, 0.04, 0.24, 0.16, 0.5], [0.2, 0.46, 0.32, 0.16, 0.04, 0.26]]


In [5]:
df2 = pd.DataFrame(data_MA, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "count_pairs"])
df2.to_csv('4_mallows_MA.csv', index=False)

In [22]:
df = pd.read_csv('4_mallows_MA.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('4_mallows_MA.csv', index=False)

**Mallows Euclidean**

In [6]:
## Euclidean Mallows Accomplice for d = 2

data_EM = []
numberEM = []
numberEM_pairs = []
n=4
phi_list=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count=[]
        count_pairs = []
        ct=[]
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            pref=generate_mallows_euclidean_votes(n, phi1, phi2)
            mpref=pref[0]
            wpref=pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt=0
                for accomplice in range(1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count_pairs.append(cnt_pairs)
            count.append(count_woman)
            data_EM.append([phi1, phi2, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count)>0:
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberEM.append(number_phi)
    numberEM_pairs.append(number_phi_pairs)
print(numberEM)
print(numberEM_pairs)

100%|██████████| 6/6 [00:00<00:00, 12.65it/s]

[[0.0, 0.0, 0.0, 0.1, 0.08, 0.0], [0.0, 0.04, 0.0, 0.12, 0.16, 0.04], [0.08, 0.12, 0.08, 0.3, 0.12, 0.08], [0.12, 0.06, 0.08, 0.2, 0.08, 0.12], [0.16, 0.08, 0.26, 0.04, 0.26, 0.08], [0.12, 0.2, 0.1, 0.2, 0.26, 0.4]]
[[0.0, 0.0, 0.0, 0.1, 0.12, 0.0], [0.0, 0.08, 0.0, 0.12, 0.2, 0.04], [0.08, 0.12, 0.12, 0.3, 0.12, 0.08], [0.12, 0.06, 0.08, 0.24, 0.08, 0.2], [0.24, 0.12, 0.26, 0.04, 0.3, 0.08], [0.12, 0.2, 0.1, 0.24, 0.34, 0.48]]


In [7]:
df2 = pd.DataFrame(data_EM, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "count_pairs"])
df2.to_csv('4_euclidean_mallows.csv', index=False)

In [19]:
df = pd.read_csv('4_euclidean_mallows.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('4_euclidean_mallows.csv', index=False)

**Reverse Euclidean**

In [8]:
#Reverse Euclidean

number_reverse_euclidean = []
number_reverse_euclidean_pairs = []
data_RE = []
n=4
p_list = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
d_list  = [2, 4, 6, 8, 10]
for d in tqdm(d_list):
    number_d = []
    number_d_pairs = []
    for p in tqdm(p_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range(50):
            count_woman = 0
            cnt_pairs = 0
            pref = generate_reverse_euclidean_votes(n, d, 'uniform', 0.5, p)
            mpref = pref[0]
            wpref = pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range (1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result[0] != matching:
                        cnt += 1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count.append(count_woman)
            count_pairs.append(cnt_pairs)
            data_RE.append([d, p, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count) > 0:
            number_d.append(sum(count)/len(count))
            number_d_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_d.append(0)
            number_d_pairs.append(0)
    number_reverse_euclidean.append(number_d)
    number_reverse_euclidean_pairs.append(number_d_pairs)
print(number_reverse_euclidean)
print(number_reverse_euclidean_pairs)

100%|██████████| 5/5 [00:00<00:00, 13.87it/s]

[[0.0, 0.2, 0.12, 0.08, 0.0, 0.0], [0.0, 0.08, 0.24, 0.22, 0.0, 0.0], [0.0, 0.04, 0.16, 0.04, 0.0, 0.0], [0.0, 0.08, 0.3, 0.12, 0.0, 0.0], [0.0, 0.2, 0.12, 0.12, 0.0, 0.0]]
[[0.0, 0.2, 0.2, 0.08, 0.0, 0.0], [0.0, 0.08, 0.32, 0.22, 0.0, 0.0], [0.0, 0.04, 0.16, 0.04, 0.0, 0.0], [0.0, 0.08, 0.38, 0.12, 0.0, 0.0], [0.0, 0.2, 0.16, 0.12, 0.0, 0.0]]


In [9]:
df3 = pd.DataFrame(data_RE, columns=["dimension", "proportion", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df3.to_csv('4_reverse_euclidean.csv', index=False)

In [20]:
df = pd.read_csv('4_reverse_euclidean.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('4_reverse_euclidean.csv', index=False)

**Mallows MD**

In [10]:
##Mallows MD

data_MD = []
numberMD = []
numberMD_pairs = []
n=4
phi_list=[0, 0.2, 0.4, 0.6, 0.8, 1]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            pref=generate_mallows_asymmetric_votes(num_agents=n, phi_m=phi1, phi_w=phi2)
            mpref=pref[0]
            wpref=pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range(1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count_pairs.append(cnt_pairs)
            count.append(count_woman)
            data_MD.append([phi1, phi2, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count) > 0: 
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberMD.append(number_phi)
    numberMD_pairs.append(number_phi_pairs) 
print(numberMD)
print(numberMD_pairs) 

100%|██████████| 6/6 [00:00<00:00, 15.02it/s]

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.06, 0.46, 0.4, 0.44, 0.2, 0.44], [1.0, 0.38, 0.54, 0.42, 0.38, 0.16], [0.8, 0.32, 0.46, 0.1, 0.4, 0.04], [0.52, 0.28, 0.38, 0.3, 0.16, 0.2], [0.4, 0.12, 0.28, 0.26, 0.14, 0.24]]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.3, 0.62, 0.44, 0.48, 0.2, 0.52], [1.16, 0.42, 0.58, 0.54, 0.5, 0.16], [0.92, 0.4, 0.46, 0.1, 0.48, 0.04], [0.68, 0.32, 0.46, 0.38, 0.24, 0.24], [0.44, 0.12, 0.44, 0.26, 0.14, 0.28]]


In [11]:
df4 = pd.DataFrame(data_MD, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df4.to_csv('4_mallows_MD.csv', index=False)

In [23]:
df = pd.read_csv('4_mallows_MD.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('4_mallows_MD.csv', index=False)

**Mallows**

In [12]:
##Mallows

numberM = []
numberM_pairs = []
data_M = []
n=4
phi_list=[0, 0.2, 0.4, 0.6, 0.8, 1]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count =[]
        count_pairs = []
        ct=[]
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            m_pref = [generate_ic_votes(n)[0]]*n
            w_pref = [generate_ic_votes(n)[0]]*n
            mpref= mallows_votes(m_pref, phi1)
            wpref= mallows_votes(w_pref, phi2)
            mpref_clean = [[int(x) for x in row] for row in mpref] #these two are for saving purposes
            wpref_clean = [[int(x) for x in row] for row in wpref]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range (1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice) 
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count.append(count_woman)
            count_pairs.append(cnt_pairs)
            data_M.append([phi1, phi2, json.dumps(mpref_clean), json.dumps(wpref_clean), samples, count_woman, cnt_pairs])
        if len(count)>0:
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberM.append(number_phi)
    numberM_pairs.append(number_phi_pairs)
print(numberM)
print(numberM_pairs)

100%|██████████| 6/6 [00:00<00:00, 14.64it/s]

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.04, 0.0], [0.0, 0.0, 0.04, 0.28, 0.12, 0.08], [0.0, 0.1, 0.04, 0.12, 0.08, 0.16], [0.0, 0.04, 0.0, 0.04, 0.36, 0.08], [0.0, 0.04, 0.08, 0.04, 0.2, 0.3]]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.04, 0.0], [0.0, 0.0, 0.04, 0.28, 0.16, 0.08], [0.0, 0.1, 0.04, 0.16, 0.12, 0.2], [0.0, 0.04, 0.0, 0.04, 0.4, 0.12], [0.0, 0.08, 0.08, 0.04, 0.24, 0.34]]


In [13]:
df5 = pd.DataFrame(data_M, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df5.to_csv('4_mallows.csv', index=False)

In [24]:
df = pd.read_csv('4_mallows.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('4_mallows.csv', index=False)

**Expectations Euclidean**

In [14]:
import numpy as np

def generate_expectation_votes(num_agents: int = None,
                               dim: int = None,
                               space='uniform',
                               std: float = None,
                               phi=0.5,
                               **kwargs):
    name = f'{dim}d_{space}'

    left_agents_reality = np.array([get_uniform_point(dim) for _ in range(num_agents)])
    left_agents_wishes = np.zeros([num_agents, dim])

    right_agents_reality = np.array([get_uniform_point(dim) for _ in range(num_agents)])
    right_agents_wishes = np.zeros([num_agents, dim])

    for v in range(num_agents):
        # while agents_wishes[v][0] <= 0 or agents_wishes[v][0] >= 1:
        left_agents_wishes[v][0] = np.random.normal(left_agents_reality[v][0], std)
        # while agents_wishes[v][1] <= 0 or agents_wishes[v][1] >= 1:
        left_agents_wishes[v][1] = np.random.normal(left_agents_reality[v][1], std)

        # while agents_wishes[v][0] <= 0 or agents_wishes[v][0] >= 1:
        right_agents_wishes[v][0] = np.random.normal(right_agents_reality[v][0], std)
        # while agents_wishes[v][1] <= 0 or agents_wishes[v][1] >= 1:
        right_agents_wishes[v][1] = np.random.normal(right_agents_reality[v][1], std)

    left_votes = np.zeros([num_agents, num_agents], dtype=int)
    right_votes = np.zeros([num_agents, num_agents], dtype=int)
    distances = np.zeros([num_agents, num_agents], dtype=float)

    for v in range(num_agents):
        for c in range(num_agents):
            left_votes[v][c] = c
            distances[v][c] = np.linalg.norm(right_agents_reality[c] - left_agents_wishes[v])
        left_votes[v] = [x for _, x in sorted(zip(distances[v], left_votes[v]))]

    for v in range(num_agents):
        for c in range(num_agents):
            right_votes[v][c] = c
            distances[v][c] = np.linalg.norm(left_agents_reality[c] - right_agents_wishes[v])
        right_votes[v] = [x for _, x in sorted(zip(distances[v], right_votes[v]))]

    left_votes_lst = [list(row) for row in left_votes]
    right_votes_lst = [list(row) for row in right_votes]


    for i in range(len(left_votes_lst)):
        for j in range(len(left_votes_lst[i])):
            left_votes_lst[i][j] = left_votes_lst[i][j] + 1
    for i in range(len(right_votes_lst)):
        for j in range(len(right_votes_lst[i])):
            right_votes_lst[i][j] = right_votes_lst[i][j] + 1

    return [left_votes_lst, right_votes_lst]

In [15]:
import numpy as np

def convert_to_builtin(obj):
    if isinstance(obj, dict):
        return {convert_to_builtin(k): convert_to_builtin(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_builtin(elem) for elem in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    else:
        return obj

In [16]:
## Expectations Euclidean

data_EE = []
number_expectations_euclidean = []
number_EE_pairs = []
n = 4
std_list = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
d_list = [2, 4, 6, 8, 10]

for d in tqdm(d_list):
    number_d = []
    number_d_pairs = []
    for std in tqdm(std_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range(50):
            count_woman = 0
            cnt_pairs = 0
            pref = generate_expectation_votes(n, d, 'uniform', std, 0.5)
            mpref = pref[0]
            wpref = pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range (1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result[0] != matching:
                        cnt += 1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count.append(count_woman)
            count_pairs.append(cnt_pairs)
            data_EE.append([d, std, json.dumps(convert_to_builtin(mpref)), json.dumps(convert_to_builtin(wpref)), samples, count_woman, cnt_pairs])
        if len(count) > 0:
            number_d.append(sum(count)/len(count))
            number_d_pairs.append(sum(count_pairs)/ len(count_pairs))
        else:
            number_d.append(0)
            number_d_pairs.append(0)
    number_expectations_euclidean.append(number_d)
    number_EE_pairs.append(number_d_pairs)
print(number_expectations_euclidean)
print(number_EE_pairs)

100%|██████████| 5/5 [00:00<00:00, 11.05it/s]

[[0.0, 0.04, 0.08, 0.32, 0.12, 0.12], [0.0, 0.0, 0.04, 0.0, 0.0, 0.04], [0.0, 0.0, 0.04, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.04], [0.0, 0.0, 0.0, 0.04, 0.0, 0.0]]
[[0.0, 0.08, 0.12, 0.48, 0.12, 0.12], [0.0, 0.0, 0.04, 0.0, 0.0, 0.04], [0.0, 0.0, 0.04, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.04], [0.0, 0.0, 0.0, 0.04, 0.0, 0.0]]


In [17]:
df6 = pd.DataFrame(data_EE, columns=["dimension", "std", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df6.to_csv('4_expectations_euclidean.csv', index=False)

In [25]:
df = pd.read_csv('4_expectations_euclidean.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('4_expectations_euclidean.csv', index=False)

In [26]:
def invert_proposer_matches(proposer_matches):
    """
    proposer_matches: list of length n where index i (0-based) is proposer (i+1)
                      and value proposer_matches[i] is the partner id (1-based).
    Returns: list `inv` of length n that maps partner -> proposer (1-based ids),
             i.e., inv[partner-1] = proposer_id
    """
    n = len(proposer_matches)
    inv = [0] * n
    for proposer_idx, partner in enumerate(proposer_matches):
        if partner == 0:
            # unmatched / sentinel (shouldn't happen in full matching)
            continue
        inv[partner - 1] = proposer_idx + 1
    return inv


def rank_differences_women(mpref, wpref, debug=False):
    """
    Return list of length n: for each woman w (1..n),
    diff[w-1] = rank_w(men-opt partner for w) - rank_w(women-opt partner for w).
    This should be >= 0 for all w (woman-optimal is at least as good).
    """
    # men-optimal: list mapping man -> woman (1-based IDs)
    men_optimal = gale_shapley(mpref, wpref)

    # women-proposing: returns list mapping woman -> man (1-based IDs)
    women_proposer_matches = gale_shapley(wpref, mpref)

    # invert to get men -> woman mapping for the women-optimal matching
    women_optimal = invert_proposer_matches(women_proposer_matches)

    n = len(mpref)
    differences = []

    for w in range(1, n + 1):
        # partner of woman w in men-optimal matching (this returns the man id who is matched to w)
        partner_men_opt = get_w_partner(w, men_optimal)   # returns man id
        # partner of woman w in women-optimal matching (man id)
        partner_women_opt = get_w_partner(w, women_optimal)

        # rank of those partners in woman's preference list (1-based ranks)
        rank_men_opt = wpref[w - 1].index(partner_men_opt) + 1
        rank_women_opt = wpref[w - 1].index(partner_women_opt) + 1

        diff = rank_men_opt - rank_women_opt
        differences.append(diff)

        if debug and diff < 0:
            print(f"DEBUG NEGATIVE: woman={w}, rank_men_opt={rank_men_opt}, "
                  f"rank_women_opt={rank_women_opt}, man_opt={partner_men_opt}, woman_opt={partner_women_opt}")
            print(" wpref[w-1] =", wpref[w-1])
            print(" men_optimal (man->woman) =", men_optimal)
            print(" women_proposer_matches (woman->man) =", women_proposer_matches)
            print(" women_optimal (man->woman) =", women_optimal)

    return differences


In [41]:
import pandas as pd
import ast
import numpy as np

# --- Your helper functions (from before) ---
# gale_shapley, get_w_partner, get_m_partner, rank_differences_women, etc.

def parse_preferences(pref_str):
    """Convert the string representation of preferences to lists."""
    return ast.literal_eval(pref_str)

def compute_differences(pref_str):
    """Compute rank differences for women for a given preference string."""
    mpref, wpref = parse_preferences(pref_str)
    return rank_differences_women(mpref, wpref)

def process_csv(filepath):
    """Load CSV, compute rank differences and average, return updated df."""
    df = pd.read_csv(filepath)
    
    # Identify the preferences column (assuming it's the 3rd column)
    df["preferences"] = df["mpref"] + "," + df["wpref"]  # Combine mpref and wpref into a single string
    pref_col = "preferences"  # Assuming the new column contains the preferences
    
    # Compute rank differences
    df["rank_differences"] = df[pref_col].apply(compute_differences)
    
    # Compute average rank difference per row
    df["avg_rank_differences"] = df["rank_differences"].apply(lambda x: np.mean(x))
    
    return df

def count_positive(lst_str):
    try:
        lst = ast.literal_eval(lst_str)  # convert string -> list
        return sum(1 for x in lst if x > 0)
    except (ValueError, SyntaxError):
        return 0  # fallback if parsing fails
    
def egalitarian_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n + 1):
        w = matching[m-1]
        cost += mpref[m-1].index(w)
        cost += wpref[w-1].index(m)
    return cost

import numpy as np


def find_all_stable_original(malechoice, femalechoice, n):
    # Output list with all matchings
    res = []
    # Initialization
    unchanged = [True] * n
    marriage = [0] * n
    malecounter = [0] * n
    # Female choice transformation

    fc = np.zeros((n, n), dtype=int)
    for i in range(1, n):
        for j in range(0, n - 1):
            fc[i][femalechoice[i][j]] = j
        fc[i][0] = n
        malecounter[i] = 0
        marriage[i] = 0

    def get_marriage(marriage):
        malematches = [0] * (n - 1)
        for j in range(1, n):
            malematches[marriage[j] - 1] = j
        return(malematches)

    def print_marriage(marriage):
        malematches = [0] * (n - 1)
        for j in range(1, n):
            malematches[marriage[j] - 1] = j
        print(*malematches)

    def proposal(i, malec, marriage, c):
        if i < 0:
            success = True
        elif (i == 0) or malec[i] == (n - 1) or not (unchanged[i]):  # n not (n+1), as we have n equals 5
            success = False
        else:
            c += 1
            j = malec[i]
            malec[i] = j + 1
            success = refusal(i, malechoice[i][j], malec, marriage, c)
        return success

    def refusal(i, j, malec, marriage, c):
        if fc[j][abs(marriage[j])] > fc[j][i]:
            k = marriage[j]
            marriage[j] = i
            success = proposal(k, malec, marriage, c)
        else:
            success = proposal(i, malec, marriage, c)
        return success

    def breakmariage(malecounter, marriage, i, n, count):
        malecounter_ori = malecounter.copy()
        marriage_ori = marriage.copy()
        marriage[malechoice[i][malecounter[i] - 1]] = -i
        success = proposal(i, malecounter, marriage, count)

        if not (success):
            unchanged[i] = False
            malecounter_ref = malecounter_ori.copy()
            marriage_ref = marriage_ori.copy()
            malecounter = malecounter_ref
            marriage = marriage_ref
            return malecounter, marriage

        #print_marriage(marriage)  # marriage[i] means the woman who is married to i man
        new_marriage = get_marriage(marriage)  # convert into female solution
        res.append(new_marriage)
        # establish the relationships between the original and the obtained stable matching

        for j in range(i, n):
            # if i==2 and j==2:
            #     print("This is it")
            malecounter, marriage = breakmariage(malecounter, marriage, j, n, count)

        for j in range((i + 1), n):
            unchanged[j] = True
        # return malecounter_ori, abs(marriage_ori)
        return malecounter_ori, marriage_ori

    count = 0
    for i in range(1, n):
        proposal(i, malecounter, marriage, count)
    #print_marriage(marriage)

    new_marriage = get_marriage(marriage)  # convert into female solution
    res.append(new_marriage)
    # establish the relationships between the original and the obtained stable matching

    M_optimal_marriage = marriage.copy()
    M_optimal_counter = malecounter.copy()
    for i in range(1, n - 1):
        breakmariage(M_optimal_counter.copy(), M_optimal_marriage.copy(), i, n, count)
    return res

def men_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n+1):
        w = matching[m-1]
        cost += mpref[m-1].index(w)
    return cost

def women_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n+1):
        w = matching[m-1]
        cost += wpref[w-1].index(m)
    return cost

In [42]:
files = [
    "4_euclidean_mallows.csv",
    "4_expectations_euclidean.csv",
    "4_mallows.csv",
    "4_mallows_MA.csv",
    "4_mallows_MD.csv",
    "4_reverse_euclidean.csv"
]
for f in files:
    df_updated = process_csv(f)
    out_file = f.replace(".csv", "_with_rank_diff.csv")
    df_updated.to_csv(out_file, index=False)
    print(f"Saved: {out_file}")

rank_gain_files = [f.replace(".csv", "_with_rank_diff.csv") for f in files]

for f in rank_gain_files:
    df = pd.read_csv(f)
    df["trunc_count"] = df["rank_differences"].apply(count_positive)
    egalitarian_matches = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref

        stable = find_all_stable_original(mpref, wpref, len(mpref))
        best_match = None
        best_cost = float("inf")
        for match in stable:
            cost = egalitarian_cost(match, m_pref, w_pref)
            if cost < best_cost:
                best_cost = cost
                best_match = match

        egalitarian_matches.append(best_match)
    df["egalitarian_match"] = egalitarian_matches
    men_matches = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref

        stable = find_all_stable_original(mpref, wpref, len(mpref))
        best_match = None
        best_cost = float("inf")
        for match in stable:
            cost = men_cost(match, m_pref, w_pref)
            if cost < best_cost:
                best_cost = cost
                best_match = match

        men_matches.append(best_match)
    df["men_opt_match"] = men_matches

    women_matches = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref

        stable = find_all_stable_original(mpref, wpref, len(mpref))
        best_match = None
        best_cost = float("inf")
        for match in stable:
            cost = women_cost(match, m_pref, w_pref)
            if cost < best_cost:
                best_cost = cost
                best_match = match

        women_matches.append(best_match)
    df["women_opt_match"] = women_matches


    cost_diffs = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref
        stable = find_all_stable_original(mpref, wpref, len(mpref))

        egal = row["egalitarian_match"]  # <- no ast.literal_eval
        egal_cost = egalitarian_cost(egal, m_pref, w_pref)

        men_opt = row["men_opt_match"]
        men_cost1 = men_cost(men_opt, m_pref, w_pref)

        women_opt = row["women_opt_match"]
        women_cost1 = women_cost(women_opt, m_pref, w_pref)

        delta = egal_cost - men_cost1 - women_cost1
        cost_diffs.append(delta)



    df["trunc_egal"] = cost_diffs 

    stability_counts = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref
        stable = find_all_stable_original(mpref, wpref, len(mpref))
        num_stable = len(stable)
        stability_counts.append(num_stable)


    df["stability_count"] = stability_counts

    df["trunc_prevalence"] = (df["stability_count"] > 1).astype(int)

    df.to_csv(f)

Saved: 4_euclidean_mallows_with_rank_diff.csv
Saved: 4_expectations_euclidean_with_rank_diff.csv
Saved: 4_mallows_with_rank_diff.csv
Saved: 4_mallows_MA_with_rank_diff.csv
Saved: 4_mallows_MD_with_rank_diff.csv
Saved: 4_reverse_euclidean_with_rank_diff.csv
